# Historical analysis notebook
Code-only copy. Outputs and original narrative cells were removed. See docs/reproduction.md for the revised analysis and execution order.


In [ ]:
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
def locate_project_root() -> Path:
    start = Path.cwd().resolve()
    for root in [start, *start.parents]:
        if (root / 'data').exists() and (root / 'results').exists() and (root / 'images').exists():
            return root
    raise FileNotFoundError('Unable to locate project root')

PROJECT_ROOT = locate_project_root()
RESULTS_DIR = PROJECT_ROOT / 'results'
IMAGES_DIR = PROJECT_ROOT / 'images'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

GROUPS = ('all_signals', 'no_gas_exchange', 'hemo_plus_nirs', 'nirs')
MODELS = ('Ridge', 'ElasticNet', 'PLS')
GROUP_DISPLAY = {
    'all_signals': 'All signals',
    'no_gas_exchange': 'No gas exchange',
    'hemo_plus_nirs': 'Hemodynamics + NIRS',
    'nirs': 'NIRS only',
}

PROJECT_ROOT

In [ ]:
def safe_name(value: str) -> str:
    return value.replace(' ', '_').replace('/', '_')

def detect_column(df: pd.DataFrame, candidates: List[str]) -> str:
    lower_map = {col.lower(): col for col in df.columns}
    for candidate in candidates:
        key = candidate.lower()
        if key in lower_map:
            return lower_map[key]
    raise KeyError(f'Could not find any candidate column in {candidates}')

def load_prediction_table(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)

    df = pd.read_csv(path)
    sid_col = detect_column(df, ['subject_id', 'Subject', 'ID'])
    actual_col = detect_column(df, ['actual_vo2peak', 'actual', 'y_true', 'VO2peak_abs'])
    pred_col = detect_column(df, ['predicted_vo2peak', 'predicted', 'y_pred', 'prediction'])

    out = df[[sid_col, actual_col, pred_col]].copy()
    out = out.rename(columns={sid_col: 'subject_id', actual_col: 'actual', pred_col: 'predicted'})
    out['residual'] = out['predicted'] - out['actual']
    out['abs_error'] = out['residual'].abs()
    out = out.dropna(subset=['subject_id', 'actual', 'predicted']).copy()
    out['subject_id'] = out['subject_id'].astype(str)
    return out.set_index('subject_id').sort_index()

def paired_stats(pre_abs: np.ndarray, post_abs: np.ndarray) -> Dict[str, float]:
    delta = post_abs - pre_abs
    out: Dict[str, float] = {
        'DeltaAbsErr_mean': float(np.mean(delta)),
        'DeltaAbsErr_median': float(np.median(delta)),
        'DeltaAbsErr_sd': float(np.std(delta, ddof=1)) if len(delta) > 1 else np.nan,
    }

    if len(delta) >= 2 and not np.allclose(delta, 0.0):
        t_stat, t_p = stats.ttest_rel(post_abs, pre_abs)
        try:
            w_stat, w_p = stats.wilcoxon(post_abs, pre_abs, zero_method='wilcox', alternative='two-sided')
        except ValueError:
            w_stat, w_p = np.nan, np.nan
        denom = np.std(delta, ddof=1)
        dz = float(np.mean(delta) / denom) if denom > 0 else np.nan
    else:
        t_stat, t_p, w_stat, w_p, dz = np.nan, np.nan, np.nan, np.nan, np.nan

    out.update({
        'PairedT_stat': float(t_stat) if not np.isnan(t_stat) else np.nan,
        'PairedT_p': float(t_p) if not np.isnan(t_p) else np.nan,
        'Wilcoxon_stat': float(w_stat) if not np.isnan(w_stat) else np.nan,
        'Wilcoxon_p': float(w_p) if not np.isnan(w_p) else np.nan,
        'Cohens_dz': dz,
    })
    return out

In [ ]:
summary_records: List[Dict[str, float]] = []
subject_records: List[Dict[str, float]] = []

for group in GROUPS:
    for model in MODELS:
        pre_path = RESULTS_DIR / f"nested_{safe_name(group)}_{safe_name(model)}_loso_predictions.csv"
        post_path = RESULTS_DIR / f"e1_zero_shot_{safe_name(group)}_{safe_name(model)}_predictions.csv"

        pre_df = load_prediction_table(pre_path)
        post_df = load_prediction_table(post_path)

        common_subjects = sorted(set(pre_df.index).intersection(post_df.index))
        if not common_subjects:
            continue

        pre_common = pre_df.loc[common_subjects].copy()
        post_common = post_df.loc[common_subjects].copy()

        paired = pd.DataFrame(index=common_subjects)
        paired['actual_pre'] = pre_common['actual']
        paired['predicted_pre'] = pre_common['predicted']
        paired['residual_pre'] = pre_common['residual']
        paired['abs_error_pre'] = pre_common['abs_error']
        paired['actual_post'] = post_common['actual']
        paired['predicted_post'] = post_common['predicted']
        paired['residual_post'] = post_common['residual']
        paired['abs_error_post'] = post_common['abs_error']
        paired['abs_error_gap_post_minus_pre'] = paired['abs_error_post'] - paired['abs_error_pre']

        pre_mae = float(paired['abs_error_pre'].mean())
        post_mae = float(paired['abs_error_post'].mean())
        pre_rmse = float(np.sqrt(np.mean(np.square(paired['residual_pre']))))
        post_rmse = float(np.sqrt(np.mean(np.square(paired['residual_post']))))

        diff = paired['abs_error_gap_post_minus_pre'].to_numpy(dtype=float)
        stats_dict = paired_stats(
            paired['abs_error_pre'].to_numpy(dtype=float),
            paired['abs_error_post'].to_numpy(dtype=float),
        )

        improved = int(np.sum(diff < -1e-12))
        worsened = int(np.sum(diff > 1e-12))
        unchanged = int(len(diff) - improved - worsened)

        summary_records.append({
            'Group': group,
            'GroupLabel': GROUP_DISPLAY[group],
            'Model': model,
            'NCommonSubjects': int(len(common_subjects)),
            'MAE_pre': pre_mae,
            'MAE_post': post_mae,
            'Delta_MAE_post_minus_pre': post_mae - pre_mae,
            'RMSE_pre': pre_rmse,
            'RMSE_post': post_rmse,
            'Delta_RMSE_post_minus_pre': post_rmse - pre_rmse,
            'Median_DeltaAbsErr': stats_dict['DeltaAbsErr_median'],
            'SD_DeltaAbsErr': stats_dict['DeltaAbsErr_sd'],
            'ImprovedN': improved,
            'WorsenedN': worsened,
            'UnchangedN': unchanged,
            'PairedT_stat': stats_dict['PairedT_stat'],
            'PairedT_p': stats_dict['PairedT_p'],
            'Wilcoxon_stat': stats_dict['Wilcoxon_stat'],
            'Wilcoxon_p': stats_dict['Wilcoxon_p'],
            'Cohens_dz': stats_dict['Cohens_dz'],
        })

        for subject_id, row in paired.iterrows():
            subject_records.append({
                'subject_id': subject_id,
                'Group': group,
                'GroupLabel': GROUP_DISPLAY[group],
                'Model': model,
                'abs_error_pre': float(row['abs_error_pre']),
                'abs_error_post': float(row['abs_error_post']),
                'abs_error_gap_post_minus_pre': float(row['abs_error_gap_post_minus_pre']),
                'residual_pre': float(row['residual_pre']),
                'residual_post': float(row['residual_post']),
            })

e2_summary_df = pd.DataFrame(summary_records)
e2_subject_df = pd.DataFrame(subject_records)

if e2_summary_df.empty:
    raise ValueError('No E2 records were generated. Check input prediction files.')

group_order = {name: idx for idx, name in enumerate(GROUPS)}
model_order = {name: idx for idx, name in enumerate(MODELS)}
e2_summary_df['GroupOrder'] = e2_summary_df['Group'].map(group_order).fillna(999)
e2_summary_df['ModelOrder'] = e2_summary_df['Model'].map(model_order).fillna(999)
e2_summary_df = e2_summary_df.sort_values(['GroupOrder', 'ModelOrder']).reset_index(drop=True)

e2_summary_path = RESULTS_DIR / 'e2_pre_post_error_gap_summary.csv'
e2_subject_path = RESULTS_DIR / 'e2_pre_post_error_gap_subject_level.csv'
e2_summary_df.to_csv(e2_summary_path, index=False)
e2_subject_df.to_csv(e2_subject_path, index=False)

display_cols = [
    'GroupLabel', 'Model', 'NCommonSubjects',
    'MAE_pre', 'MAE_post', 'Delta_MAE_post_minus_pre',
    'RMSE_pre', 'RMSE_post', 'Delta_RMSE_post_minus_pre',
    'PairedT_p', 'Wilcoxon_p', 'Cohens_dz',
    'ImprovedN', 'WorsenedN', 'UnchangedN',
]

display(e2_summary_df[display_cols])
print('Saved:', e2_summary_path)
print('Saved:', e2_subject_path)

In [ ]:
table_full = e2_summary_df[[
    'GroupLabel', 'Model', 'NCommonSubjects',
    'MAE_pre', 'MAE_post', 'Delta_MAE_post_minus_pre',
    'RMSE_pre', 'RMSE_post', 'Delta_RMSE_post_minus_pre',
    'PairedT_p', 'Wilcoxon_p', 'Cohens_dz',
    'ImprovedN', 'WorsenedN', 'UnchangedN',
]].rename(columns={
    'GroupLabel': 'Group',
    'NCommonSubjects': 'N',
    'Delta_MAE_post_minus_pre': 'Delta MAE (post-pre)',
    'Delta_RMSE_post_minus_pre': 'Delta RMSE (post-pre)',
    'PairedT_p': 'Paired t p',
    'Wilcoxon_p': 'Wilcoxon p',
})

table_full_csv = RESULTS_DIR / 'e2_results_table_full.csv'
table_full_tex = RESULTS_DIR / 'e2_results_table_full.tex'
table_full.to_csv(table_full_csv, index=False)

table_tex_df = table_full.copy()
for col in ['MAE_pre', 'MAE_post', 'Delta MAE (post-pre)', 'RMSE_pre', 'RMSE_post', 'Delta RMSE (post-pre)', 'Cohens_dz']:
    table_tex_df[col] = table_tex_df[col].map(lambda x: round(float(x), 3))
for col in ['Paired t p', 'Wilcoxon p']:
    table_tex_df[col] = table_full[col].map(lambda x: '<0.001' if pd.notna(x) and float(x) < 0.001 else (f'{float(x):.3f}' if pd.notna(x) else 'NA'))

table_tex_df.to_latex(
    table_full_tex,
    index=False,
    escape=False,
    caption='Experiment 2: paired pre-vs-post error-gap analysis for all feature-group/model combinations.',
    label='tab:e2_full',
)

best_worst = pd.concat([
    e2_summary_df.sort_values('Delta_MAE_post_minus_pre').head(2),
    e2_summary_df.sort_values('Delta_MAE_post_minus_pre').tail(2),
]).drop_duplicates().reset_index(drop=True)

best_worst_table = best_worst[[
    'GroupLabel', 'Model', 'NCommonSubjects',
    'MAE_pre', 'MAE_post', 'Delta_MAE_post_minus_pre',
    'PairedT_p', 'Wilcoxon_p', 'Cohens_dz',
]].rename(columns={
    'GroupLabel': 'Group',
    'NCommonSubjects': 'N',
    'Delta_MAE_post_minus_pre': 'Delta MAE (post-pre)',
    'PairedT_p': 'Paired t p',
    'Wilcoxon_p': 'Wilcoxon p',
})

best_worst_csv = RESULTS_DIR / 'e2_results_table_key_configs.csv'
best_worst_tex = RESULTS_DIR / 'e2_results_table_key_configs.tex'
best_worst_table.to_csv(best_worst_csv, index=False)

best_worst_tex_df = best_worst_table.copy()
for col in ['MAE_pre', 'MAE_post', 'Delta MAE (post-pre)', 'Cohens_dz']:
    best_worst_tex_df[col] = best_worst_tex_df[col].map(lambda x: round(float(x), 3))
for col in ['Paired t p', 'Wilcoxon p']:
    best_worst_tex_df[col] = best_worst_table[col].map(lambda x: '<0.001' if pd.notna(x) and float(x) < 0.001 else (f'{float(x):.3f}' if pd.notna(x) else 'NA'))

best_worst_tex_df.to_latex(
    best_worst_tex,
    index=False,
    escape=False,
    caption='Experiment 2: key configurations with smallest and largest MAE gap (post-pre).',
    label='tab:e2_key_configs',
)

display(table_tex_df)
print('Saved:', table_full_csv)
print('Saved:', table_full_tex)
print('Saved:', best_worst_csv)
print('Saved:', best_worst_tex)

In [ ]:
combo_order = [f"{GROUP_DISPLAY[g]}\n{m}" for g in GROUPS for m in MODELS]
combo_df = e2_summary_df.copy()
combo_df['ComboLabel'] = combo_df['GroupLabel'] + '\n' + combo_df['Model']

# Figure 1: Delta MAE heatmap
delta_matrix = e2_summary_df.pivot(index='Group', columns='Model', values='Delta_MAE_post_minus_pre').reindex(GROUPS)[list(MODELS)]
delta_vals = delta_matrix.to_numpy(dtype=float)
fig, ax = plt.subplots(figsize=(7.2, 3.4))
im = ax.imshow(delta_vals, cmap='coolwarm', aspect='auto')
ax.set_xticks(np.arange(len(MODELS)), labels=MODELS)
ax.set_yticks(np.arange(len(GROUPS)), labels=[GROUP_DISPLAY[g] for g in GROUPS])
for i in range(delta_vals.shape[0]):
    for j in range(delta_vals.shape[1]):
        value = delta_vals[i, j]
        txt_color = 'white' if abs(value) > np.nanmedian(np.abs(delta_vals)) else 'black'
        ax.text(j, i, f'{value:+.3f}', ha='center', va='center', color=txt_color, fontsize=9)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Delta MAE (post - pre) [L/min]')
ax.set_title('E2 error gap heatmap')
ax.set_xlabel('Model')
ax.set_ylabel('Feature group')
fig.tight_layout()
fig1_path = IMAGES_DIR / 'e2_delta_mae_heatmap.png'
fig.savefig(fig1_path, dpi=300, bbox_inches='tight')
plt.close(fig)

# Figure 2: Pre vs post MAE by combination
plot_df = combo_df.sort_values(['GroupOrder', 'ModelOrder']).reset_index(drop=True)
x = np.arange(plot_df.shape[0])
width = 0.38
fig, ax = plt.subplots(figsize=(12, 4.2))
ax.bar(x - width / 2, plot_df['MAE_pre'], width, label='Pre (baseline LOSO)')
ax.bar(x + width / 2, plot_df['MAE_post'], width, label='Post (zero-shot follow-up)')
ax.set_xticks(x, labels=plot_df['ComboLabel'], rotation=35, ha='right')
ax.set_ylabel('MAE [L/min]')
ax.set_title('E2 pre vs post MAE per combination')
ax.legend(loc='upper left')
fig.tight_layout()
fig2_path = IMAGES_DIR / 'e2_pre_post_mae_by_combo.png'
fig.savefig(fig2_path, dpi=300, bbox_inches='tight')
plt.close(fig)

# Figure 3: Subject-level abs error gap distribution by combination
e2_subject_df['ComboLabel'] = e2_subject_df['GroupLabel'] + '\n' + e2_subject_df['Model']
combo_labels = plot_df['ComboLabel'].tolist()
box_data = [
    e2_subject_df.loc[e2_subject_df['ComboLabel'] == label, 'abs_error_gap_post_minus_pre'].to_numpy(dtype=float)
    for label in combo_labels
]
fig, ax = plt.subplots(figsize=(12, 4.2))
ax.boxplot(box_data, tick_labels=combo_labels, showfliers=True)
ax.axhline(0.0, color='black', linestyle='--', linewidth=1)
ax.set_xticklabels(combo_labels, rotation=35, ha='right')
ax.set_ylabel('Subject-level delta abs error (post - pre) [L/min]')
ax.set_title('E2 subject-level error-gap distributions')
fig.tight_layout()
fig3_path = IMAGES_DIR / 'e2_subject_level_error_gap_boxplot.png'
fig.savefig(fig3_path, dpi=300, bbox_inches='tight')
plt.close(fig)

print('Saved:', fig1_path)
print('Saved:', fig2_path)
print('Saved:', fig3_path)

In [ ]:
def fmt_p(p: float) -> str:
    if pd.isna(p):
        return 'NA'
    return '<0.001' if float(p) < 0.001 else f'{float(p):.3f}'

e2_sorted = e2_summary_df.sort_values('Delta_MAE_post_minus_pre').reset_index(drop=True)
best_gap = e2_sorted.iloc[0]
worst_gap = e2_sorted.iloc[-1]

n_degrade = int((e2_summary_df['Delta_MAE_post_minus_pre'] > 0).sum())
n_improve = int((e2_summary_df['Delta_MAE_post_minus_pre'] < 0).sum())
mean_delta_mae = float(e2_summary_df['Delta_MAE_post_minus_pre'].mean())

paragraph_1 = (
    f"Experiment 2 quantified model transport degradation by comparing baseline LOSO error (pre) "
    f"against zero-shot follow-up error (post) on the same participants ($n=15$ per configuration). "
    f"Across all 12 feature-group/model combinations, the mean MAE gap was "
    f"{mean_delta_mae:+.3f}~L$\\cdot$min$^{{-1}}$ (post minus pre), with "
    f"{n_degrade}/12 combinations showing error increase and {n_improve}/12 showing error reduction "
    f"(Table~\\ref{{tab:e2_full}})."
 )

paragraph_2 = (
    f"The smallest MAE gap was observed for {best_gap['GroupLabel']} + {best_gap['Model']} "
    f"(MAE pre={best_gap['MAE_pre']:.3f}, post={best_gap['MAE_post']:.3f}, "
    f"$\\Delta$MAE={best_gap['Delta_MAE_post_minus_pre']:+.3f}~L$\\cdot$min$^{{-1}}$, "
    f"paired $p$={fmt_p(best_gap['PairedT_p'])}; Wilcoxon $p$={fmt_p(best_gap['Wilcoxon_p'])}). "
    f"The largest gap was observed for {worst_gap['GroupLabel']} + {worst_gap['Model']} "
    f"(MAE pre={worst_gap['MAE_pre']:.3f}, post={worst_gap['MAE_post']:.3f}, "
    f"$\\Delta$MAE={worst_gap['Delta_MAE_post_minus_pre']:+.3f}~L$\\cdot$min$^{{-1}}$)."
 )

paragraph_3 = (
    "Figure~\\ref{fig:e2_delta_heatmap} summarizes the MAE-gap landscape across all configurations. "
    "Figure~\\ref{fig:e2_pre_post_mae} contrasts pre and post MAE directly, and "
    "Figure~\\ref{fig:e2_subject_gap_box} shows subject-level heterogeneity of paired error gaps."
 )

section_text = (
    "\\subsection{E2: Pre-vs-post prediction error gap}\n\n"
    + paragraph_1 + "\n\n"
    + paragraph_2 + "\n\n"
    + paragraph_3 + "\n"
 )

figures_text = """\\begin{figure}[t]
\\centering
\\includegraphics[width=0.95\\linewidth]{images/e2_delta_mae_heatmap.png}
\\caption{Experiment 2 MAE-gap heatmap across feature groups and models. Positive values indicate worse post-transfer performance.}
\\label{fig:e2_delta_heatmap}
\\end{figure}

\\begin{figure}[t]
\\centering
\\includegraphics[width=0.95\\linewidth]{images/e2_pre_post_mae_by_combo.png}
\\caption{Pre (baseline LOSO) versus post (zero-shot follow-up) MAE per feature-group/model configuration.}
\\label{fig:e2_pre_post_mae}
\\end{figure}

\\begin{figure}[t]
\\centering
\\includegraphics[width=0.95\\linewidth]{images/e2_subject_level_error_gap_boxplot.png}
\\caption{Subject-level distribution of paired absolute-error gaps (post minus pre) for each configuration.}
\\label{fig:e2_subject_gap_box}
\\end{figure}
"""

bundle_text = (
    section_text
    + "\n% Auto-generated tables\n"
    + "\\input{results/e2_results_table_full.tex}\n"
    + "\\input{results/e2_results_table_key_configs.tex}\n\n"
    + figures_text
 )

section_path = RESULTS_DIR / 'e2_results_section.tex'
bundle_path = RESULTS_DIR / 'e2_results_with_tables_and_figures.tex'
section_path.write_text(section_text, encoding='utf-8')
bundle_path.write_text(bundle_text, encoding='utf-8')

print(section_text)
print('Saved:', section_path)
print('Saved:', bundle_path)